In [2]:
import pandas as pd
import numpy as np

# 재현성 고정
np.random.seed(42)

# 주문 수 고정
N_ORDERS = 5000

# 완제품 수 고정
N_PRODUCTS = 20

# 완제품 ID 생성
product_ids = [f"PRD{str(i).zfill(3)}" for i in range(1, N_PRODUCTS+1)]

# 주문 ID 생성
order_ids = [f"ORD{str(i).zfill(6)}" for i in range(1, N_ORDERS+1)]

# 주문별 제품 랜덤 선택
weights = np.random.dirichlet(np.ones(N_PRODUCTS))

selected_products = np.random.choice(
    product_ids,
    size=N_ORDERS,
    p=weights
)

# 주문 수량 생성
order_qty = np.random.normal(loc=50, scale=20, size=N_ORDERS)
order_qty = order_qty.astype(int)
order_qty = np.clip(order_qty, 5, 150)

# 주문 일자 생성
start_date = pd.to_datetime("2025-08-28")

order_dates = start_date + pd.to_timedelta(
    np.random.randint(0, 100, size=N_ORDERS),
    unit="D"
)

# 납기일 생성
base_lead = np.random.randint(10, 20, size=N_ORDERS)

urgency = np.random.choice(
    [0, -3, -7],
    size=N_ORDERS,
    p=[0.7, 0.2, 0.1]
)

due_dates = order_dates + pd.to_timedelta(base_lead + urgency, unit="D")

# orders 생성
orders = pd.DataFrame({
    "order_id" : order_ids,
    "product_id" : selected_products,
    "order_qty" : order_qty,
    "order_date" : order_dates,
    "due_date" : due_dates
})

# CSV로 내보내기
try:
    orders.to_csv("orders.csv", index=False, encoding="utf-8-sig")
    print("orders.csv 생성 완료:", orders.shape)
    print(orders.head())
except Exception as e:
    print("생성 실패:", e)

orders.csv 생성 완료: (5000, 5)
    order_id product_id  order_qty order_date   due_date
0  ORD000001     PRD012         35 2025-11-19 2025-11-27
1  ORD000002     PRD002         87 2025-10-17 2025-11-04
2  ORD000003     PRD004         50 2025-08-28 2025-09-08
3  ORD000004     PRD008         65 2025-09-23 2025-10-11
4  ORD000005     PRD009         37 2025-11-02 2025-11-16


In [3]:
import pandas as pd

df = pd.read_csv("orders.csv")

print("===== 기본 정보 =====")
print("shape:", df.shape)
print("columns:", df.columns.tolist())

print("\n===== PK 체크 =====")
print("order_id 중복:", df["order_id"].duplicated().sum())

print("\n===== 제품 분포 =====")
print(df["product_id"].value_counts().head())

print("\n===== 수량 분포 =====")
print(df["order_qty"].describe())

print("\n===== 날짜 체크 =====")
df["order_date"] = pd.to_datetime(df["order_date"])
df["due_date"] = pd.to_datetime(df["due_date"])

print("납기 오류:", (df["due_date"] < df["order_date"]).sum())

lead_days = (df["due_date"] - df["order_date"]).dt.days
print("\n===== 납기 여유 =====")
print(lead_days.describe())

===== 기본 정보 =====
shape: (5000, 5)
columns: ['order_id', 'product_id', 'order_qty', 'order_date', 'due_date']

===== PK 체크 =====
order_id 중복: 0

===== 제품 분포 =====
product_id
PRD012    983
PRD002    858
PRD008    572
PRD013    490
PRD003    352
Name: count, dtype: int64

===== 수량 분포 =====
count    5000.000000
mean       49.230000
std        20.084157
min         5.000000
25%        35.750000
50%        49.000000
75%        63.000000
max       120.000000
Name: order_qty, dtype: float64

===== 날짜 체크 =====
납기 오류: 0

===== 납기 여유 =====
count    5000.000000
mean       13.083200
std         3.699867
min         3.000000
25%        11.000000
50%        13.000000
75%        16.000000
max        19.000000
dtype: float64
